In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
log_df = pd.read_csv("log.csv")
trace_df = pd.read_csv("trace.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df, ts_col in [(metric_df, "timestamp"), (log_df, "timestamp"), (trace_df, "timestamp"), (error_logs_df, "timestamp")]:
    df[ts_col] = pd.to_datetime(df[ts_col], unit="s", utc=True)

# --- 1) metric.csv summary ---
# Ensure value column numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

metric_group = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count_of_points=('value', 'count'),
        global_P95_of_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
        min_timestamp_utc=('timestamp', 'min'),
        max_timestamp_utc=('timestamp', 'max'),
    )
    .sort_values('count_of_points', ascending=False)
)

# Format timestamp columns to compact UTC string
metric_group[['min_timestamp_utc','max_timestamp_utc']] = metric_group[['min_timestamp_utc','max_timestamp_utc']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

metric_summary_head = metric_group.head(20)

# --- 2) log.csv summary ---
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

log_group = (
    log_df
    .groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count_of_points=('value', 'count'),
        global_P95_of_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
        min_timestamp_utc=('timestamp', 'min'),
        max_timestamp_utc=('timestamp', 'max'),
    )
    .sort_values('count_of_points', ascending=False)
)

log_group[['min_timestamp_utc','max_timestamp_utc']] = log_group[['min_timestamp_utc','max_timestamp_utc']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

log_summary_head = log_group.head(20)

# --- 3) trace.csv summary ---
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

trace_group = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count_of_points=('value', 'count'),
        global_P95_of_value=('value', lambda s: float(s.quantile(0.95)) if s.notna().any() else float('nan')),
        min_timestamp_utc=('timestamp', 'min'),
        max_timestamp_utc=('timestamp', 'max'),
    )
    .sort_values('count_of_points', ascending=False)
)

trace_group[['min_timestamp_utc','max_timestamp_utc']] = trace_group[['min_timestamp_utc','max_timestamp_utc']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

trace_summary_head = trace_group.head(20)

# --- 4) error_logs.csv summary ---
error_group = (
    error_logs_df
    .groupby('cmdb_id', as_index=False)
    .agg(
        count_of_error_rows=('message', 'count'),
        min_timestamp_utc=('timestamp', 'min'),
        max_timestamp_utc=('timestamp', 'max'),
    )
    .sort_values('count_of_error_rows', ascending=False)
)

error_group[['min_timestamp_utc','max_timestamp_utc']] = error_group[['min_timestamp_utc','max_timestamp_utc']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

error_summary_head = error_group.head(20)

# --- Overall min/max timestamps per file ---
overall_bounds = pd.DataFrame([
    {"file":"metric.csv", "min_timestamp_utc": metric_df['timestamp'].min(), "max_timestamp_utc": metric_df['timestamp'].max()},
    {"file":"log.csv",    "min_timestamp_utc": log_df['timestamp'].min(),    "max_timestamp_utc": log_df['timestamp'].max()},
    {"file":"trace.csv",  "min_timestamp_utc": trace_df['timestamp'].min(),  "max_timestamp_utc": trace_df['timestamp'].max()},
    {"file":"error_logs.csv","min_timestamp_utc": error_logs_df['timestamp'].min(), "max_timestamp_utc": error_logs_df['timestamp'].max()},
])
overall_bounds[['min_timestamp_utc','max_timestamp_utc']] = overall_bounds[['min_timestamp_utc','max_timestamp_utc']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

file_overall_bounds_df = overall_bounds

# --- Incident window and quick metric coverage check ---
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)

metric_in_window = metric_df[(metric_df['timestamp'] >= incident_start) & (metric_df['timestamp'] <= incident_end)].copy()

metric_window_overall_count = metric_in_window.shape[0]

metric_window_group = (
    metric_in_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        points_in_window=('value','count'),
        min_ts_in_window=('timestamp','min'),
        max_ts_in_window=('timestamp','max'),
        max_value_in_window=('value','max'),
    )
    .sort_values('points_in_window', ascending=False)
)

# format window timestamps
if not metric_window_group.empty:
    metric_window_group[['min_ts_in_window','max_ts_in_window']] = metric_window_group[['min_ts_in_window','max_ts_in_window']].apply(lambda s: s.dt.strftime('%Y-%m-%d %H:%M:%S%z'))

metric_window_summary_head = metric_window_group.head(20)

# Final compact outputs (variables displayed)
metric_summary_head, log_summary_head, trace_summary_head, error_summary_head, file_overall_bounds_df, metric_window_overall_count, metric_window_summary_head

```
Out[1]:
```
import pandas as pd

# Reuse existing kernel variables: metric_df, log_df, trace_df, error_logs_df, metric_group, log_group, trace_group, error_group, metric_window_overall_count, incident_start, incident_end
# Build a plain-English summary string based on the previously computed results.

metric_min = metric_df['timestamp'].min()
metric_max = metric_df['timestamp'].max()
log_min = log_df['timestamp'].min()
log_max = log_df['timestamp'].max()
trace_min = trace_df['timestamp'].min()
trace_max = trace_df['timestamp'].max()
error_rows_count = error_logs_df.shape[0]

# top components present in metric summary (top by count)
top_metric_cmdb = metric_group.head(10)['cmdb_id'].unique().tolist()
top_metric_cmdb_short = ", ".join(top_metric_cmdb[:6])

# coverage offsets relative to incident window
start_offset = int((metric_min - incident_start).total_seconds())
end_offset = int((incident_end - metric_max).total_seconds())

summary = (
    "Summary of telemetry coverage and quick findings (plain English):\n\n"
    f"- Metric telemetry: present from {metric_min.strftime('%Y-%m-%d %H:%M:%S%Z')} to {metric_max.strftime('%Y-%m-%d %H:%M:%S%Z')} UTC.\n"
    f"  This is {abs(start_offset)} seconds after the incident window start (so telemetry begins ~{abs(start_offset)//60}m{abs(start_offset)%60}s after 2024-01-23 16:29:06) "
    f"and ends {abs(end_offset)} seconds before the incident window end (so telemetry stops ~{abs(end_offset)//60}m{abs(end_offset)%60}s before 2024-01-23 16:59:06).\n\n"
    f"- Log telemetry: present from {log_min.strftime('%Y-%m-%d %H:%M:%S%Z')} to {log_max.strftime('%Y-%m-%d %H:%M:%S%Z')} UTC.\n"
    f"- Trace telemetry: present from {trace_min.strftime('%Y-%m-%d %H:%M:%S%Z')} to {trace_max.strftime('%Y-%m-%d %H:%M:%S%Z')} UTC.\n"
    f"- Error log rows: {error_rows_count} (no distinct cmdb_id rows were found in error_logs.csv for this dataset).\n\n"
    f"- Within the incident window (2024-01-23 16:29:06 to 2024-01-23 16:59:06 UTC), metric points found: {metric_window_overall_count}.\n"
    f"  Metric data in the window is available for many services; top observed services in the metric summary include: {top_metric_cmdb_short} (and others).\n\n"
    "Implication for next steps:\n"
    "- We have metric, log, and trace telemetry covering a central portion of the incident window (from 16:32:00 to 16:56:00 UTC). Use these metrics/traces for anomaly detection.\n"
    "- There are no error_logs entries to support a log-based error root-cause. Proceed with metric- and trace-based anomaly analysis for the listed services.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id    kpi_name  count_of_points  global_P95_of_value         min_timestamp_utc         max_timestamp_utc
370           ts-voucher-service      socket               25         4.000000e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
0    ts-admin-basic-info-service         cpu               25         4.679087e-01  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
1    ts-admin-basic-info-service      diskio               25         1.229621e+06  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
2    ts-admin-basic-info-service  latency-50               25         1.335977e-02  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
3    ts-admin-basic-info-service  latency-90               25         3.029531e-02  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
4    ts-admin-basic-info-service         mem               25         2.350894e+08  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
5    ts-admin-basic-info-service      socket               25         7.666667e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
6    ts-admin-basic-info-service    workload               25         6.983167e-01  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
7         ts-admin-order-service         cpu               25         1.059915e-01  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
8         ts-admin-order-service      diskio               25         4.414845e+04  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
9         ts-admin-order-service         mem               25         2.007307e+08  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
10        ts-admin-order-service      socket               25         3.403333e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
11        ts-admin-route-service         cpu               25         8.228280e-02  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
12        ts-admin-route-service      diskio               25         4.397714e+04  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
13        ts-admin-route-service         mem               25         1.920349e+08  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
14        ts-admin-route-service      socket               25         3.406667e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
15       ts-admin-travel-service         cpu               25         4.439090e-01  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
16       ts-admin-travel-service      diskio               25         1.199905e+06  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
17       ts-admin-travel-service       error               25         0.000000e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
331             ts-travel2-mongo      socket               25         6.000000e+00  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000,                  cmdb_id         log_name  count_of_points  global_P95_of_value         min_timestamp_utc         max_timestamp_utc
6   ts-assurance-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
7   ts-assurance-service    log.row_count               25                 14.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
10       ts-auth-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
11       ts-auth-service    log.row_count               25                344.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
12      ts-basic-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
13      ts-basic-service    log.row_count               25               2639.2  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
30   ts-food-map-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
31   ts-food-map-service    log.row_count               25                 26.8  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
27   ts-contacts-service    log.row_count               25                 52.8  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
26   ts-contacts-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
63      ts-price-service    log.row_count               25                291.2  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
62      ts-price-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
91    ts-travel2-service    log.row_count               25               2384.2  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
90    ts-travel2-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
93       ts-ui-dashboard    log.row_count               25                291.2  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
92       ts-ui-dashboard  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
67      ts-route-service    log.row_count               25               1064.2  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
66      ts-route-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
69       ts-seat-service    log.row_count               25               1957.4  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
68       ts-seat-service  log.error_count               25                  0.0  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000,                 cmdb_id                                    trace_name  count_of_points  global_P95_of_value         min_timestamp_utc         max_timestamp_utc
736  ts-travel2-service        trace.to_ts-seat-service.duration_mean               25             0.057781  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
737  ts-travel2-service         trace.to_ts-seat-service.duration_p95               25             0.107264  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
738  ts-travel2-service           trace.to_ts-seat-service.error_rate               25             0.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
739  ts-travel2-service            trace.to_ts-seat-service.row_count               25           164.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
740  ts-travel2-service  trace.to_ts-ticketinfo-service.duration_mean               25             0.018543  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
741  ts-travel2-service   trace.to_ts-ticketinfo-service.duration_p95               25             0.050265  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
742  ts-travel2-service     trace.to_ts-ticketinfo-service.error_rate               25             0.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
743  ts-travel2-service      trace.to_ts-ticketinfo-service.row_count               25           708.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
744  ts-travel2-service       trace.to_ts-train-service.duration_mean               25             0.005250  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
745  ts-travel2-service        trace.to_ts-train-service.duration_p95               25             0.005702  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
746  ts-travel2-service          trace.to_ts-train-service.error_rate               25             0.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
747  ts-travel2-service           trace.to_ts-train-service.row_count               25           298.800000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
748  ts-travel2-service     trace.to_ts-travel2-service.duration_mean               25             0.012241  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
749  ts-travel2-service      trace.to_ts-travel2-service.duration_p95               25             0.053097  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
750  ts-travel2-service        trace.to_ts-travel2-service.error_rate               25             0.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
751  ts-travel2-service         trace.to_ts-travel2-service.row_count               25          2721.800000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
760     ts-user-service      trace.from_ts-user-service.duration_mean               25             0.017372  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
761     ts-user-service       trace.from_ts-user-service.duration_p95               25             0.021707  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
762     ts-user-service         trace.from_ts-user-service.error_rate               25             0.000000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
763     ts-user-service          trace.from_ts-user-service.row_count               25            17.600000  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000, Empty DataFrame
Columns: [cmdb_id, count_of_error_rows, min_timestamp_utc, max_timestamp_utc]
Index: [],              file         min_timestamp_utc         max_timestamp_utc
0      metric.csv  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
1         log.csv  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
2       trace.csv  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
3  error_logs.csv                       NaN                       NaN, 9216,                          cmdb_id    kpi_name  points_in_window          min_ts_in_window          max_ts_in_window  max_value_in_window
370           ts-voucher-service      socket                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         4.000000e+00
0    ts-admin-basic-info-service         cpu                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         4.988716e-01
1    ts-admin-basic-info-service      diskio                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.239705e+06
2    ts-admin-basic-info-service  latency-50                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.381358e-02
3    ts-admin-basic-info-service  latency-90                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         3.289086e-02
4    ts-admin-basic-info-service         mem                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         2.351164e+08
5    ts-admin-basic-info-service      socket                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         7.883333e+00
6    ts-admin-basic-info-service    workload                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         7.078000e-01
7         ts-admin-order-service         cpu                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.113432e-01
8         ts-admin-order-service      diskio                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         6.123486e+04
9         ts-admin-order-service         mem                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         2.007310e+08
10        ts-admin-order-service      socket                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         3.433333e+00
11        ts-admin-route-service         cpu                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.137219e-01
12        ts-admin-route-service      diskio                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         4.577155e+04
13        ts-admin-route-service         mem                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.920352e+08
14        ts-admin-route-service      socket                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         3.433333e+00
15       ts-admin-travel-service         cpu                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         4.721333e-01
16       ts-admin-travel-service      diskio                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         1.309427e+06
17       ts-admin-travel-service       error                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         0.000000e+00
331             ts-travel2-mongo      socket                25  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000         6.000000e+00)```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing dataframes from kernel: metric_df, log_df, trace_df
# If they are not present, load them (defensive)
try:
    metric_df
    log_df
    trace_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    log_df = pd.read_csv("log.csv")
    trace_df = pd.read_csv("trace.csv")
    for df, ts_col in [(metric_df, "timestamp"), (log_df, "timestamp"), (trace_df, "timestamp")]:
        df[ts_col] = pd.to_datetime(df[ts_col], unit="s", utc=True)

candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

def component_stats(df, file_name, candidates):
    rows = []
    for comp in candidates:
        sub = df[df['cmdb_id'] == comp]
        cnt = int(sub.shape[0])
        if cnt > 0:
            min_ts = sub['timestamp'].min()
            max_ts = sub['timestamp'].max()
            min_ts_s = min_ts.strftime('%Y-%m-%d %H:%M:%S%z')
            max_ts_s = max_ts.strftime('%Y-%m-%d %H:%M:%S%z')
        else:
            min_ts_s = pd.NA
            max_ts_s = pd.NA
        rows.append({
            'component_name': comp,
            'file_name': file_name,
            'count_of_rows_in_file_for_component': cnt,
            'min_timestamp_utc': min_ts_s,
            'max_timestamp_utc': max_ts_s
        })
    return pd.DataFrame(rows)

metric_stats_df = component_stats(metric_df, 'metric.csv', candidates)
log_stats_df = component_stats(log_df, 'log.csv', candidates)
trace_stats_df = component_stats(trace_df, 'trace.csv', candidates)

combined_df = pd.concat([metric_stats_df, log_stats_df, trace_stats_df], ignore_index=True)

# Determine which candidates are missing from all telemetry files
agg_counts = combined_df.groupby('component_name', as_index=False)['count_of_rows_in_file_for_component'].sum()
missing_components = agg_counts[agg_counts['count_of_rows_in_file_for_component'] == 0]['component_name'].tolist()

if missing_components:
    missing_summary = "Missing from all telemetry files: " + ", ".join(missing_components)
else:
    missing_summary = "All candidate components appear in at least one telemetry file."

# Keep output compact: show combined_df and the missing_summary
combined_df, missing_summary

```
Out[2]:
```
from textwrap import dedent

# Reuse combined_df and missing_summary from kernel state
candidates = ["ts-auth-service", "ts-order-service", "ts-route-service", "ts-train-service", "ts-travel-service"]

lines = []
for comp in candidates:
    comp_rows = combined_df[combined_df['component_name'] == comp]
    metric_row = comp_rows[comp_rows['file_name'] == 'metric.csv'].iloc[0]
    log_row = comp_rows[comp_rows['file_name'] == 'log.csv'].iloc[0]
    trace_row = comp_rows[comp_rows['file_name'] == 'trace.csv'].iloc[0]
    lines.append(f"- {comp}:")
    lines.append(f"    metric.csv -> rows: {metric_row['count_of_rows_in_file_for_component']}, time range: {metric_row['min_timestamp_utc']} to {metric_row['max_timestamp_utc']}")
    lines.append(f"    log.csv    -> rows: {log_row['count_of_rows_in_file_for_component']}, time range: {log_row['min_timestamp_utc']} to {log_row['max_timestamp_utc']}")
    lines.append(f"    trace.csv  -> rows: {trace_row['count_of_rows_in_file_for_component']}, time range: {trace_row['min_timestamp_utc']} to {trace_row['max_timestamp_utc']}")
    # a short observation about sparsity
    if int(log_row['count_of_rows_in_file_for_component']) < 10:
        lines.append(f"    NOTE: log rows for {comp} are sparse ({log_row['count_of_rows_in_file_for_component']} rows).")
    if int(trace_row['count_of_rows_in_file_for_component']) == max(combined_df[combined_df['file_name']=='trace.csv']['count_of_rows_in_file_for_component']):
        lines.append(f"    NOTE: {comp} has the largest trace coverage among candidates.")

summary = dedent(
    "Telemetry presence for candidate components (compact):\n\n" +
    "\n".join(lines) + "\n\n" +
    f"{missing_summary}"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       component_name   file_name  count_of_rows_in_file_for_component         min_timestamp_utc         max_timestamp_utc
0     ts-auth-service  metric.csv                                  175  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
1    ts-order-service  metric.csv                                  200  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
2    ts-route-service  metric.csv                                  182  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
3    ts-train-service  metric.csv                                  185  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
4   ts-travel-service  metric.csv                                  200  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
5     ts-auth-service     log.csv                                   50  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
6    ts-order-service     log.csv                                   50  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
7    ts-route-service     log.csv                                   50  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
8    ts-train-service     log.csv                                    2  2024-01-23 16:47:00+0000  2024-01-23 16:47:00+0000
9   ts-travel-service     log.csv                                   50  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
10    ts-auth-service   trace.csv                                  300  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
11   ts-order-service   trace.csv                                  880  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
12   ts-route-service   trace.csv                                  500  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
13   ts-train-service   trace.csv                                  500  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000
14  ts-travel-service   trace.csv                                 1188  2024-01-23 16:32:00+0000  2024-01-23 16:56:00+0000, 'All candidate components appear in at least one telemetry file.')```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df if available; otherwise load and parse
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Preprocessing / Threshold calculation (global P95 per (cmdb_id,kpi_name) for the five candidates)
candidate_metric_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

global_p95_df = (
    candidate_metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'global_P95'})
)

# 2) Extract incident-window data (inclusive)
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)

window_df = candidate_metric_df[
    (candidate_metric_df['timestamp'] >= incident_start) &
    (candidate_metric_df['timestamp'] <= incident_end)
].copy()

# 3) Anomaly summarization
# Attach global_P95 to window rows to compute per-row above-threshold flags
window_with_thresh = window_df.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')
window_with_thresh['above_P95'] = window_with_thresh['value'] > window_with_thresh['global_P95']

# Aggregate within window
window_agg = (
    window_with_thresh
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        points_in_window = ('value','count'),
        points_above_P95 = ('above_P95','sum'),
        earliest_timestamp_above_P95 = ('timestamp', lambda s: s[window_with_thresh.loc[s.index, 'above_P95']].min() if s.size>0 else pd.NaT),
        max_value_in_window = ('value','max'),
    )
)

# There may be (cmdb_id,kpi_name) pairs that exist in global_p95_df but had zero rows in window.
# Start from global_p95_df and left-join the window aggregates to include those pairs.
detailed_results_df = global_p95_df.merge(window_agg, on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0 and keep timestamps as NaT where missing
detailed_results_df['points_in_window'] = detailed_results_df['points_in_window'].fillna(0).astype(int)
detailed_results_df['points_above_P95'] = detailed_results_df['points_above_P95'].fillna(0).astype(int)
detailed_results_df['max_value_in_window'] = detailed_results_df['max_value_in_window']  # keep NaN if no data
detailed_results_df['earliest_timestamp_above_P95'] = pd.to_datetime(detailed_results_df['earliest_timestamp_above_P95'])

# Compute max_vs_threshold_ratio, handle global_P95 == 0
detailed_results_df['max_vs_threshold_ratio'] = detailed_results_df['max_value_in_window'] / detailed_results_df['global_P95']
detailed_results_df.loc[detailed_results_df['global_P95'] == 0, 'max_vs_threshold_ratio'] = np.nan

# Format earliest timestamp above P95 as UTC string or null
def fmt_ts(x):
    if pd.isna(x):
        return pd.NA
    return x.strftime('%Y-%m-%d %H:%M:%S%z')

detailed_results_df['earliest_timestamp_above_P95'] = detailed_results_df['earliest_timestamp_above_P95'].apply(fmt_ts)

# Also format global_P95 and max_value_in_window to reasonable numeric precision
detailed_results_df['global_P95'] = detailed_results_df['global_P95'].astype(float)
detailed_results_df['max_value_in_window'] = detailed_results_df['max_value_in_window'].astype(float)

# Final summary table: select requested columns and sort
summary_cols = [
    'cmdb_id',
    'kpi_name',
    'global_P95',
    'points_in_window',
    'points_above_P95',
    'earliest_timestamp_above_P95',
    'max_value_in_window',
    'max_vs_threshold_ratio'
]

summary_table = (
    detailed_results_df[summary_cols]
    .sort_values(
        by=['points_above_P95','max_vs_threshold_ratio','cmdb_id'],
        ascending=[False, False, True]
    )
    .head(20)
    .reset_index(drop=True)
)

# One-line note: how many pairs had at least one point_above_P95 in the incident window
pairs_with_anomaly_count = int((detailed_results_df['points_above_P95'] > 0).sum())
note = f"Number of (cmdb_id,kpi_name) pairs with at least one point above global P95 in the incident window: {pairs_with_anomaly_count}"

# Persisted variable: detailed_results_df (full per-pair results) is stored in kernel as required.
detailed_results_df, summary_table, note

```
Out[3]:
```
Summary (plain English):

- Coverage: metric data for all five candidate services is present across the incident window (25 points per KPI in-window, from 2024-01-23 16:32:00 to 2024-01-23 16:56:00 UTC).

- Anomalies detected: 36 distinct (cmdb_id, kpi_name) pairs had at least one value above their global P95 during the incident window.

- Most notable spikes (highest max / P95 ratios and earliest exceedance time):
  1. ts-order-service — latency-90: earliest exceedance 2024-01-23 16:46:00 UTC, max_vs_threshold_ratio ≈ 7.08 (large latency spike).
  2. ts-travel-service — cpu: earliest exceedance 2024-01-23 16:35:00 UTC, max_vs_threshold_ratio ≈ 2.47 (CPU spike).
  3. ts-train-service — latency-90: earliest exceedance 2024-01-23 16:47:00 UTC, max_vs_threshold_ratio ≈ 2.01.
  4. ts-train-service — cpu: earliest exceedance 2024-01-23 16:47:00 UTC, max_vs_threshold_ratio ≈ 1.83.
  5. ts-route-service — latency-90: earliest exceedance 2024-01-23 16:50:00 UTC, max_vs_threshold_ratio ≈ 1.51.

- Overall pattern: each affected KPI typically shows 2 points above its global P95 within the window (points_in_window = 25, points_above_P95 = 2 for the listed top rows). Many KPIs across all five services show modest exceedances (ratios ~1.03–1.3), while a few show strong spikes (ratios >2, especially ts-order-service latency-90).

Actionable takeaway:
- Investigate ts-order-service latency (latency-90) first because it shows the largest relative spike.
- Also investigate ts-travel-service CPU and ts-train-service latency/cpu as secondary suspects.
- Because anomalies are present across multiple services, check upstream dependencies, load patterns, and any shared infrastructure (network, DB) that could produce widespread but varying-severity spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_P95  points_in_window  points_above_P95 earliest_timestamp_above_P95  max_value_in_window  max_vs_threshold_ratio
0     ts-auth-service         cpu  3.957092e+01                25                 2     2024-01-23 16:46:00+0000         3.977216e+01                1.005086
1     ts-auth-service      diskio  2.345220e+07                25                 2     2024-01-23 16:53:00+0000         2.410872e+07                1.027994
2     ts-auth-service  latency-50  1.448499e+00                25                 2     2024-01-23 16:51:00+0000         1.535056e+00                1.059756
3     ts-auth-service  latency-90  2.467519e+00                25                 2     2024-01-23 16:55:00+0000         2.919545e+00                1.183190
4     ts-auth-service         mem  6.577960e+08                25                 2     2024-01-23 16:52:00+0000         6.989274e+08                1.062529
..                ...         ...           ...               ...               ...                          ...                  ...                     ...
34  ts-travel-service  latency-50  1.366259e-02                25                 2     2024-01-23 16:38:00+0000         1.422799e-02                1.041383
35  ts-travel-service  latency-90  2.895291e-01                25                 2     2024-01-23 16:38:00+0000         2.987704e-01                1.031918
36  ts-travel-service         mem  2.746418e+08                25                 2     2024-01-23 16:38:00+0000         2.760515e+08                1.005133
37  ts-travel-service      socket  1.519333e+01                25                 2     2024-01-23 16:36:00+0000         1.648333e+01                1.084906
38  ts-travel-service    workload  1.090023e+01                25                 2     2024-01-23 16:35:00+0000         1.164438e+01                1.068269

[39 rows x 8 columns],               cmdb_id    kpi_name    global_P95  points_in_window  points_above_P95 earliest_timestamp_above_P95  max_value_in_window  max_vs_threshold_ratio
0    ts-order-service  latency-90  3.527283e-02                25                 2     2024-01-23 16:46:00+0000         2.497479e-01                7.080461
1   ts-travel-service         cpu  3.044239e+00                25                 2     2024-01-23 16:35:00+0000         7.508838e+00                2.466573
2    ts-train-service  latency-90  9.259708e-03                25                 2     2024-01-23 16:47:00+0000         1.861329e-02                2.010137
3    ts-train-service         cpu  3.753449e+00                25                 2     2024-01-23 16:47:00+0000         6.863966e+00                1.828709
4    ts-route-service  latency-90  8.772349e-03                25                 2     2024-01-23 16:50:00+0000         1.328934e-02                1.514913
5   ts-travel-service       error  2.464333e-02                25                 2     2024-01-23 16:38:00+0000         3.220000e-02                1.306641
6    ts-order-service         cpu  2.961412e+00                25                 2     2024-01-23 16:35:00+0000         3.820238e+00                1.290006
7     ts-auth-service  latency-90  2.467519e+00                25                 2     2024-01-23 16:55:00+0000         2.919545e+00                1.183190
8    ts-route-service      diskio  1.326287e+06                25                 2     2024-01-23 16:52:00+0000         1.455903e+06                1.097728
9   ts-travel-service      socket  1.519333e+01                25                 2     2024-01-23 16:36:00+0000         1.648333e+01                1.084906
10  ts-travel-service      diskio  5.977438e+04                25                 2     2024-01-23 16:32:00+0000         6.476355e+04                1.083467
11   ts-route-service         cpu  7.434035e+00                25                 2     2024-01-23 16:50:00+0000         7.997104e+00                1.075742
12  ts-travel-service    workload  1.090023e+01                25                 2     2024-01-23 16:35:00+0000         1.164438e+01                1.068269
13   ts-order-service  latency-50  9.391263e-03                25                 2     2024-01-23 16:35:00+0000         1.001089e-02                1.065980
14    ts-auth-service         mem  6.577960e+08                25                 2     2024-01-23 16:52:00+0000         6.989274e+08                1.062529
15    ts-auth-service  latency-50  1.448499e+00                25                 2     2024-01-23 16:51:00+0000         1.535056e+00                1.059756
16   ts-route-service  latency-50  3.637677e-03                25                 2     2024-01-23 16:50:00+0000         3.815621e-03                1.048917
17  ts-travel-service  latency-50  1.366259e-02                25                 2     2024-01-23 16:38:00+0000         1.422799e-02                1.041383
18   ts-train-service    workload  2.213213e+01                25                 2     2024-01-23 16:38:00+0000         2.301893e+01                1.040069
19   ts-order-service    workload  7.687537e+00                25                 2     2024-01-23 16:35:00+0000         7.988817e+00                1.039191, 'Number of (cmdb_id,kpi_name) pairs with at least one point above global P95 in the incident window: 36')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing kernel variables where possible
# Expecting: metric_df, global_p95_df, window_df may exist from prior steps
# Defensive loading/parsing if not present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Ensure global_p95_df (global P95 per cmdb_id,kpi_name for candidates) exists; otherwise compute from metric_df for the five candidates
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
if 'global_p95_df' not in globals():
    candidate_metric_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
    global_p95_df = (
        candidate_metric_df
        .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value':'global_P95'})
    )

# Ensure window_df (incident-window rows for candidate metrics) exists; otherwise compute
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)

if 'window_df' not in globals():
    candidate_metric_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()
    window_df = candidate_metric_df[
        (candidate_metric_df['timestamp'] >= incident_start) &
        (candidate_metric_df['timestamp'] <= incident_end)
    ].copy()

# Build window_with_thresh (attach global P95 and flag above_P95)
window_with_thresh = window_df.merge(global_p95_df, on=['cmdb_id','kpi_name'], how='left')
window_with_thresh['value'] = pd.to_numeric(window_with_thresh['value'], errors='coerce')
window_with_thresh['global_P95'] = pd.to_numeric(window_with_thresh['global_P95'], errors='coerce')
window_with_thresh['above_P95'] = window_with_thresh['value'] > window_with_thresh['global_P95']

# Filter only rows above threshold to detect segments
above = window_with_thresh[window_with_thresh['above_P95']].copy()

# If no above-threshold rows, create empty segments_df
if above.empty:
    segments_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','segment_start_utc','segment_end_utc',
        'num_points_in_segment','max_value_in_segment','global_P95','max_vs_threshold_ratio'
    ])
    top_segments = segments_df.head(20)
    summary_note = "Total segments detected: 0. Segments filtered out by rule: 0."
else:
    # Sort for consecutive detection
    above = above.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # Compute gaps within each (cmdb_id,kpi_name)
    ts_diff = above.groupby(['cmdb_id','kpi_name'])['timestamp'].diff()
    # A new segment starts if diff is NaT (first row) or greater than 1 minute
    new_segment = ts_diff.isna() | (ts_diff > pd.Timedelta(minutes=1))
    # Build segment index within each group by cumulative sum of new_segment per group
    above['segment_idx'] = new_segment.groupby([above['cmdb_id'], above['kpi_name']]).cumsum().astype(int)

    # Aggregate per segment
    seg_agg = (
        above
        .groupby(['cmdb_id','kpi_name','segment_idx'], as_index=False)
        .agg(
            segment_start_utc = ('timestamp','min'),
            segment_end_utc   = ('timestamp','max'),
            num_points_in_segment = ('value','count'),
            max_value_in_segment = ('value','max'),
            global_P95 = ('global_P95','first')
        )
    )

    # Compute ratio, handle global_P95 == 0
    seg_agg['max_vs_threshold_ratio'] = seg_agg['max_value_in_segment'] / seg_agg['global_P95']
    seg_agg.loc[seg_agg['global_P95'] == 0, 'max_vs_threshold_ratio'] = np.nan

    # Format timestamps to UTC strings
    seg_agg['segment_start_utc'] = seg_agg['segment_start_utc'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    seg_agg['segment_end_utc']   = seg_agg['segment_end_utc'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

    # Step 3: filter out segments with num_points == 1 AND max_vs_threshold_ratio <= 1.5
    total_segments = seg_agg.shape[0]
    filter_mask = (seg_agg['num_points_in_segment'] == 1) & (seg_agg['max_vs_threshold_ratio'] <= 1.5)
    filtered_out_count = int(filter_mask.sum())
    segments_df = seg_agg[~filter_mask].copy().reset_index(drop=True)

    # Order and produce compact top-20
    top_segments = segments_df.sort_values(
        by=['max_vs_threshold_ratio','num_points_in_segment'],
        ascending=[False, False]
    ).head(20).reset_index(drop=True)

    summary_note = f"Total segments detected: {total_segments}. Segments filtered out by rule: {filtered_out_count}."

# Persist full segments_df in kernel (already assigned). Return compact top-20 summary and the one-line summary note.
top_segments, summary_note

```
Out[4]:
```
summary = (
    "Fault-identification summary (plain English):\n\n"
    "- Total above-P95 segments detected across the five candidate services during the incident window: 63.\n"
    "- Segments filtered out (single-point and low-severity, ratio <= 1.5): 51.\n"
    "- Remaining significant segments after filtering: 12.\n\n"
    "Top detected segments (highest severity first):\n"
    "1) ts-order-service | latency-90 — single-point at 2024-01-23 16:46:00 UTC — max/P95 ≈ 7.08 (largest relative spike).\n"
    "2) ts-travel-service | cpu — single-point at 2024-01-23 16:38:00 UTC — max/P95 ≈ 2.47 (notable CPU spike).\n"
    "3) ts-train-service | latency-90 — single-point at 2024-01-23 16:47:00 UTC — max/P95 ≈ 2.01.\n"
    "4) ts-train-service | cpu — 2-point segment from 2024-01-23 16:47:00 to 16:48:00 UTC — max/P95 ≈ 1.83.\n"
    "5) ts-route-service | latency-90 — 2-point segment from 2024-01-23 16:50:00 to 16:51:00 UTC — max/P95 ≈ 1.51.\n\n"
    "Observations and action items:\n"
    "- The largest relative anomaly is ts-order-service latency-90; investigate ts-order-service latency first (likely the most impactful).\n"
    "- Secondary investigation targets: ts-travel-service CPU and ts-train-service (latency and CPU), since they show multi-point or high-ratio spikes.\n"
    "- Many detected segments were isolated, low-severity single-point spikes (51 filtered out), suggesting the event includes a mix of brief spikes and a few true high-severity incidents.\n"
    "- Next steps: correlate the retained segments with traces and logs for those timestamps (especially the top 5 above) to find root-cause (upstream dependency, DB, or network).\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  segment_idx         segment_start_utc           segment_end_utc  num_points_in_segment  max_value_in_segment    global_P95  max_vs_threshold_ratio
0    ts-order-service  latency-90            1  2024-01-23 16:46:00+0000  2024-01-23 16:46:00+0000                      1          2.497479e-01  3.527283e-02                7.080461
1   ts-travel-service         cpu            2  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000                      1          7.508838e+00  3.044239e+00                2.466573
2    ts-train-service  latency-90            1  2024-01-23 16:47:00+0000  2024-01-23 16:47:00+0000                      1          1.861329e-02  9.259708e-03                2.010137
3    ts-train-service         cpu            1  2024-01-23 16:47:00+0000  2024-01-23 16:48:00+0000                      2          6.863966e+00  3.753449e+00                1.828709
4    ts-route-service  latency-90            1  2024-01-23 16:50:00+0000  2024-01-23 16:51:00+0000                      2          1.328934e-02  8.772349e-03                1.514913
5     ts-auth-service  latency-90            1  2024-01-23 16:55:00+0000  2024-01-23 16:56:00+0000                      2          2.919545e+00  2.467519e+00                1.183190
6    ts-route-service         cpu            1  2024-01-23 16:50:00+0000  2024-01-23 16:51:00+0000                      2          7.997104e+00  7.434035e+00                1.075742
7     ts-auth-service    workload            1  2024-01-23 16:32:00+0000  2024-01-23 16:33:00+0000                      2          2.876611e+00  2.855050e+00                1.007552
8    ts-train-service      socket            1  2024-01-23 16:45:00+0000  2024-01-23 16:46:00+0000                      2          1.693333e+01  1.681000e+01                1.007337
9    ts-route-service         mem            1  2024-01-23 16:50:00+0000  2024-01-23 16:51:00+0000                      2          2.617457e+08  2.600752e+08                1.006423
10   ts-order-service         mem            1  2024-01-23 16:55:00+0000  2024-01-23 16:56:00+0000                      2          2.859622e+08  2.855544e+08                1.001428
11   ts-train-service         mem            1  2024-01-23 16:55:00+0000  2024-01-23 16:56:00+0000                      2          2.568970e+08  2.568928e+08                1.000017, 'Total segments detected: 63. Segments filtered out by rule: 51.')```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse trace_df if present, otherwise load and parse
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Parse timestamp as UTC and ensure numeric value
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Top suspect services
suspects = ["ts-order-service", "ts-travel-service", "ts-train-service"]

# 1) Compute global P95 for each distinct trace_name using full series (global, not windowed)
trace_global_p95 = (
    trace_df
    .groupby('trace_name', as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'global_P95'})
)

# 2) Filter trace rows to the incident window (inclusive)
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)

trace_window = trace_df[(trace_df['timestamp'] >= incident_start) & (trace_df['timestamp'] <= incident_end)].copy()

# 3) Select rows matching either cmdb_id in suspects OR trace_name containing relevant substrings
substrs = [
    'to_ts-order-service','to_ts-travel-service','to_ts-train-service',
    'from_ts-order-service','from_ts-travel-service','from_ts-train-service'
]
pattern = "|".join(substrs)

match_mask = (
    trace_window['cmdb_id'].isin(suspects) |
    trace_window['trace_name'].str.contains(pattern, regex=True, na=False)
)

matching_trace_rows = trace_window[match_mask].copy()

# Persist matching_trace_rows and trace_global_p95 for later cross-analysis (kernel variables)
# 4) Attach global_P95 to matching rows
matching_trace_with_p95 = matching_trace_rows.merge(trace_global_p95, on='trace_name', how='left')

# Ensure numeric and compute above flag
matching_trace_with_p95['global_P95'] = pd.to_numeric(matching_trace_with_p95['global_P95'], errors='coerce')
matching_trace_with_p95['above_P95'] = matching_trace_with_p95['value'] > matching_trace_with_p95['global_P95']

# 5) Aggregate per (cmdb_id, trace_name)
def earliest_above(series_ts, series_flag):
    # series_ts and series_flag are aligned Series
    if series_flag.any():
        return series_ts[series_flag].min()
    return pd.NaT

grouped = matching_trace_with_p95.groupby(['cmdb_id','trace_name'], as_index=False)

summary_agg = grouped.agg(
    global_P95 = ('global_P95','first'),
    points_in_window = ('value','count'),
    points_above_P95 = ('above_P95','sum'),
    max_value_in_window = ('value','max'),
)

# Compute earliest_timestamp_above_P95 separately to allow conditional min
earliest_list = []
for (cmdb, tname), grp in grouped:
    ts_min = earliest_above(grp['timestamp'], grp['above_P95'])
    earliest_list.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'earliest_timestamp_above_P95': ts_min
    })
earliest_df = pd.DataFrame(earliest_list)

# Merge earliest timestamps
summary_df = summary_agg.merge(earliest_df, on=['cmdb_id','trace_name'], how='left')

# Compute max_vs_threshold_ratio, handle global_P95 == 0
summary_df['max_vs_threshold_ratio'] = summary_df['max_value_in_window'] / summary_df['global_P95']
summary_df.loc[summary_df['global_P95'] == 0, 'max_vs_threshold_ratio'] = np.nan

# Format earliest timestamp to UTC string or NA
def fmt_ts(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(x).strftime('%Y-%m-%d %H:%M:%S%z')

summary_df['earliest_timestamp_above_P95'] = summary_df['earliest_timestamp_above_P95'].apply(fmt_ts)

# Order by points_above_P95 desc, then max_vs_threshold_ratio desc, limit to top 20
summary_table = (
    summary_df[
        ['cmdb_id','trace_name','global_P95','points_in_window','points_above_P95',
         'earliest_timestamp_above_P95','max_value_in_window','max_vs_threshold_ratio']
    ]
    .sort_values(by=['points_above_P95','max_vs_threshold_ratio'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# One-line summary: how many (cmdb_id,trace_name) pairs had points_above_P95 > 0 among selected traces
num_pairs_with_anomaly = int((summary_df['points_above_P95'] > 0).sum())
summary_note = f"Number of (cmdb_id, trace_name) pairs with points_above_P95 > 0 among selected traces: {num_pairs_with_anomaly}"

# Expose variables in kernel for further analysis:
# - matching_trace_rows (incident-window raw rows matching selection)
# - trace_global_p95 (global P95 per trace_name)
# - matching_trace_with_p95 (matching rows with attached global_P95 and above flag)
# - full_trace_summary_df (full per-pair summary before head(20))
full_trace_summary_df = summary_df.copy()

# Final compact outputs
summary_table, summary_note

```
Out[5]:
```
Plain-English summary of trace analysis for ts-order-service, ts-travel-service, ts-train-service:

- Coverage & overall signal:
  - Among the selected traces and services, 44 distinct (cmdb_id, trace_name) pairs showed at least one incident-window value above their global P95.
  - Many elevated trace signals involve calls to/from the three suspect services (especially traces referencing ts-order-service and ts-travel-service).

- Most important observations (prioritized):
  1. Elevated traffic into ts-order-service:
     - ts-order-service — trace.to_ts-order-service.row_count: 25 points in window, 9 points above P95 (earliest 2024-01-23 16:33:00 UTC), max ratio ≈ 1.42. This indicates significantly increased request volume toward order-service during the window.
     - Related traces (trace.to_ts-station-service.* and root → to_ts-order-service.*) also show 9 points above P95 (duration_p95 and duration_mean), with earliest exceedances around 16:45 UTC — indicating increased latencies around the same times that order-related activity spikes.
  2. ts-travel-service involvement:
     - trace.from_ts-preserve-service.duration_p95 and duration_mean for ts-travel-service show many points above P95 (11–13 points above), with earliest exceedance 16:32:00 UTC and max ratios up to ≈1.77 — travel-service is seeing sustained elevated durations from preserve-service calls.
     - ts-travel-service row_count traces (to/from) show 8 points above P95 (earliest ~16:33:00), indicating increased throughput around travel-service as well.
     - Additional travel-related traces (from_ts-seat-service.duration_p95, from_ts-food-service.duration_mean) show strong relative spikes (ratios ≈2.25 and ≈2.05).
  3. ts-train-service related traces:
     - trace.to_ts-train-service.row_count (source cmdb_id = ts-train-service) has 5 points above P95 (earliest 16:33:00), modest ratio ≈1.08.
     - Notably, a trace row from a different service (cmdb_id = ts-basic-service → trace.to_ts-train-service.duration_p95) shows a very large ratio ≈10.6 (3 points above P95, earliest 16:46:00). This is a high-severity signal on a trace path involving train-service and deserves immediate attention despite smaller point count.
  4. Timing pattern:
     - Earliest trace anomalies cluster around two windows: roughly 16:32–16:37 UTC and a second cluster around 16:45–16:49 UTC. This aligns with the metric-window spikes found earlier.

- Implication for root-cause drilling:
  - Strong candidates to investigate first:
    1. ts-order-service (increased incoming row_count + elevated durations at ~16:45 UTC) — aligns with the largest metric latency spike seen earlier.
    2. ts-travel-service (sustained elevated durations and row counts starting ~16:32 UTC).
    3. The specific high-ratio trace from ts-basic-service → ts-train-service (ratio ~10.6 at ~16:46 UTC) — although fewer points, the relative spike is large and could indicate localized failures affecting train-related calls.
  - Because both volume (row_count) and latency/duration traces are elevated for these services and their neighbors, investigate:
    - Upstream callers (preserve-service, station-service, basic-service, seat/food services) at the timestamps identified.
    - Shared dependencies (DB, cache, network) that could cause simultaneous increases in latency/row_count across multiple services.
    - Correlate the top trace timestamps (16:32–16:37 and ~16:45–16:49 UTC) with metric spikes and any available logs for error/timeout patterns.

- One-line summary:
  - 44 (cmdb_id, trace_name) pairs had points above global P95 among the selected traces during the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id                                    trace_name   global_P95  points_in_window  points_above_P95 earliest_timestamp_above_P95  max_value_in_window  max_vs_threshold_ratio
0   ts-travel-service  trace.from_ts-preserve-service.duration_mean     0.270158                23                13     2024-01-23 16:32:00+0000             0.339355                1.256139
1   ts-travel-service   trace.from_ts-preserve-service.duration_p95     0.287782                23                11     2024-01-23 16:32:00+0000             0.508598                1.767303
2    ts-order-service           trace.to_ts-order-service.row_count   428.500000                25                 9     2024-01-23 16:33:00+0000           607.000000                1.416569
3    ts-order-service      trace.to_ts-station-service.duration_p95     0.967185                24                 9     2024-01-23 16:45:00+0000             1.344041                1.389643
4                root        trace.to_ts-order-service.duration_p95     0.980218                24                 9     2024-01-23 16:45:00+0000             1.338799                1.365819
5    ts-order-service     trace.to_ts-station-service.duration_mean     0.894654                24                 9     2024-01-23 16:45:00+0000             1.049593                1.173184
6                root       trace.to_ts-order-service.duration_mean     0.916012                24                 9     2024-01-23 16:45:00+0000             1.054256                1.150919
7     ts-seat-service     trace.from_ts-travel-service.duration_p95     0.066676                25                 8     2024-01-23 16:37:00+0000             0.114259                1.713655
8   ts-travel-service        trace.from_ts-travel-service.row_count  1468.100000                25                 8     2024-01-23 16:33:00+0000          1949.000000                1.327566
9   ts-travel-service          trace.to_ts-travel-service.row_count  1468.400000                25                 8     2024-01-23 16:33:00+0000          1949.000000                1.327295
10               root       trace.to_ts-travel-service.duration_p95     0.915168                25                 8     2024-01-23 16:37:00+0000             1.196106                1.306980
11    ts-seat-service    trace.from_ts-travel-service.duration_mean     0.044945                25                 8     2024-01-23 16:37:00+0000             0.057637                1.282390
12               root      trace.to_ts-travel-service.duration_mean     0.551371                25                 8     2024-01-23 16:37:00+0000             0.664433                1.205057
13   ts-train-service           trace.to_ts-train-service.row_count  2536.600000                25                 5     2024-01-23 16:33:00+0000          2748.000000                1.083340
14  ts-travel-service       trace.from_ts-seat-service.duration_p95     0.031928                25                 4     2024-01-23 16:37:00+0000             0.071815                2.249268
15  ts-travel-service      trace.from_ts-food-service.duration_mean     0.013575                25                 4     2024-01-23 16:44:00+0000             0.027880                2.053737
16   ts-order-service  trace.from_ts-security-service.duration_mean     0.016906                25                 4     2024-01-23 16:37:00+0000             0.026831                1.587057
17   ts-order-service                  trace.from_root.duration_p95     1.204133                24                 4     2024-01-23 16:49:00+0000             1.338799                1.111836
18   ts-order-service                 trace.from_root.duration_mean     1.004027                24                 4     2024-01-23 16:49:00+0000             1.054256                1.050027
19   ts-basic-service        trace.to_ts-train-service.duration_p95     0.005780                25                 3     2024-01-23 16:46:00+0000             0.061316               10.607466, 'Number of (cmdb_id, trace_name) pairs with points_above_P95 > 0 among selected traces: 44')```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing variables from kernel: segments_df, trace_df, trace_global_p95, trace_window, matching_trace_with_p95
# Defensive checks and (re)computations if needed
try:
    segments_df
except NameError:
    raise NameError("segments_df not found in kernel. Ensure prior segments computation was run.")

# Ensure trace_df and trace_global_p95 exist
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
if 'trace_global_p95' not in globals():
    trace_global_p95 = (
        trace_df
        .groupby('trace_name', as_index=False)['value']
        .quantile(0.95)
        .rename(columns={'value':'global_P95'})
    )

# Ensure trace_window (incident-window subset) exists or compute it
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)
if 'trace_window' not in globals():
    trace_window = trace_df[(trace_df['timestamp'] >= incident_start) & (trace_df['timestamp'] <= incident_end)].copy()

# Prepare segments_df: convert segment start/end to timestamps if they're strings
seg = segments_df.copy()
# If segment_start_utc is string, parse; otherwise assume datetime-like
seg['segment_start_ts'] = pd.to_datetime(seg['segment_start_utc'], utc=True, errors='coerce')
seg['segment_end_ts'] = pd.to_datetime(seg['segment_end_utc'], utc=True, errors='coerce')

# Select top 12 significant segments by severity (max_vs_threshold_ratio desc, then num_points_in_segment desc)
top12_segments = seg.sort_values(by=['max_vs_threshold_ratio','num_points_in_segment'], ascending=[False, False]).head(12).reset_index(drop=True)

# For each segment, find matching trace rows: timestamp within [start,end] AND (trace cmdb_id == seg cmdb_id OR trace_name contains 'to_<cmdb_id>' or 'from_<cmdb_id>')
matches = []
for _, row in top12_segments.iterrows():
    s_cmdb = row['cmdb_id']
    s_kpi = row['kpi_name']
    s_start = row['segment_start_ts']
    s_end = row['segment_end_ts']
    if pd.isna(s_start) or pd.isna(s_end):
        continue
    # Filter trace_window by timestamp range first (inclusive)
    tw = trace_window[(trace_window['timestamp'] >= s_start) & (trace_window['timestamp'] <= s_end)].copy()
    if tw.empty:
        continue
    # Build substring patterns for to_/from_
    to_sub = f"to_{s_cmdb}"
    from_sub = f"from_{s_cmdb}"
    # Mask for cmdb match or trace_name containing substrings
    mask = (tw['cmdb_id'] == s_cmdb) | tw['trace_name'].str.contains(to_sub, na=False) | tw['trace_name'].str.contains(from_sub, na=False)
    matched_traces = tw[mask].copy()
    if matched_traces.empty:
        continue
    # Attach global P95 for the trace_name
    matched_traces = matched_traces.merge(trace_global_p95, on='trace_name', how='left')
    matched_traces['global_P95'] = pd.to_numeric(matched_traces['global_P95'], errors='coerce')
    matched_traces['trace_value_above_P95'] = matched_traces['value'] > matched_traces['global_P95']
    matched_traces['trace_max_vs_threshold_ratio'] = matched_traces['value'] / matched_traces['global_P95']
    matched_traces.loc[matched_traces['global_P95'] == 0, 'trace_max_vs_threshold_ratio'] = np.nan

    # For each matched trace row, produce the required output row
    out = matched_traces[['timestamp','cmdb_id','trace_name','value','global_P95','trace_value_above_P95','trace_max_vs_threshold_ratio']].copy()
    out = out.rename(columns={
        'timestamp':'trace_timestamp_utc',
        'cmdb_id':'trace_cmdb_id',
        'value':'trace_value'
    })
    out['segment_cmdb_id'] = s_cmdb
    out['segment_kpi_name'] = s_kpi
    out['segment_start_utc'] = s_start
    # Reorder columns to match requested order
    out = out[[
        'segment_cmdb_id','segment_kpi_name','segment_start_utc',
        'trace_timestamp_utc','trace_cmdb_id','trace_name','trace_value',
        'global_P95','trace_value_above_P95','trace_max_vs_threshold_ratio'
    ]]
    matches.append(out)

# Concatenate all matches into a single DataFrame (full matching join table)
if matches:
    full_matching_trace_correlations = pd.concat(matches, ignore_index=True)
else:
    full_matching_trace_correlations = pd.DataFrame(columns=[
        'segment_cmdb_id','segment_kpi_name','segment_start_utc',
        'trace_timestamp_utc','trace_cmdb_id','trace_name','trace_value',
        'global_P95','trace_value_above_P95','trace_max_vs_threshold_ratio'
    ])

# Format segment_start_utc and trace_timestamp_utc as UTC strings for compact display
if not full_matching_trace_correlations.empty:
    full_matching_trace_correlations['segment_start_utc'] = pd.to_datetime(full_matching_trace_correlations['segment_start_utc'], utc=True).dt.strftime('%Y-%m-%d %H:%M:%S%z')
    full_matching_trace_correlations['trace_timestamp_utc'] = pd.to_datetime(full_matching_trace_correlations['trace_timestamp_utc'], utc=True).dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Build compact summary: order by segment_start_utc asc, then trace_value_above_P95 True first, then trace_max_vs_threshold_ratio desc; limit top 50
if not full_matching_trace_correlations.empty:
    # For ordering, convert segment_start_utc back to datetime for sort (but keep string for display)
    sort_df = full_matching_trace_correlations.copy()
    sort_df['_seg_start_dt'] = pd.to_datetime(sort_df['segment_start_utc'], utc=True)
    sort_df = sort_df.sort_values(by=['_seg_start_dt','trace_value_above_P95','trace_max_vs_threshold_ratio'], ascending=[True, False, False])
    compact_summary = sort_df.drop(columns=['_seg_start_dt']).head(50).reset_index(drop=True)
else:
    compact_summary = full_matching_trace_correlations.copy()

# One-line counts
total_segments_processed = top12_segments.shape[0]
total_matching_trace_rows_found = int(full_matching_trace_correlations.shape[0])
total_trace_rows_above = int(full_matching_trace_correlations['trace_value_above_P95'].sum()) if not full_matching_trace_correlations.empty else 0
counts_note = (
    f"Total segments processed: {total_segments_processed}. "
    f"Total matching trace rows found: {total_matching_trace_rows_found}. "
    f"Trace rows with value > global_P95: {total_trace_rows_above}."
)

# Expose full_matching_trace_correlations in kernel for later use (already assigned)
# Return the compact summary (limited) and the counts note
compact_summary, counts_note

```
Out[6]:
```
Summary (plain English):

- Scope and counts:
  - Processed the top 12 retained metric segments.
  - Found 736 matching trace rows in the incident window that align with those segments.
  - Of those trace rows, 52 had values above their trace-specific global P95.

- High-level interpretation:
  - Correlations are present but relatively sparse: only ~7% (52/736) of matching trace rows exceeded their P95. This indicates many trace measurements coincide with the metric segments, but a smaller subset are clear trace anomalies.
  - The strongest and most actionable correlations match the same services identified earlier from metrics and trace summary:

- Top correlated findings to investigate first:
  1) ts-order-service — latency spike at 2024-01-23 16:46 UTC
     - Correlated with multiple trace anomalies on paths into order-service: elevated to_ts-order-service.row_count and increased duration_p95/duration_mean around ~16:45–16:46 UTC.
     - This aligns metric latency spike and elevated incoming traffic — prioritize ts-order-service and its callers (station-service, root callers) around 16:45–16:46 UTC.
  2) ts-travel-service — CPU spike at ~16:38 UTC
     - Correlated with elevated traces from preserve-service (duration_p95/mean) and elevated to/from travel-service row_count starting ~16:32–16:38 UTC.
     - Suggests sustained load/duration increases impacting travel-service beginning ~16:32 and peaking ~16:38.
  3) ts-train-service — latency/cpu around 16:47 UTC
     - Notable high-severity trace: a ts-basic-service → ts-train-service duration_p95 shows a very large ratio (~10.6) at ~16:46 UTC (high relative spike). Even though point counts are small, this is a strong signal for train-related call failures or timeouts.
  4) ts-auth-service — workload spikes at ~16:32–16:33 UTC
     - Several trace rows (e.g., trace.from_root.row_count) around 16:32–16:33 slightly exceed P95; less severe but present.

- Practical next steps:
  - Investigate ts-order-service first (traffic + latency overlap at ~16:45–16:46).
  - Correlate the highlighted trace timestamps (especially ~16:32–16:38 and ~16:45–16:49 UTC) with logs and DB/infra metrics (network, DB, cache) to find upstream cause.
  - Pay special attention to the ts-basic-service → ts-train-service high-ratio trace at ~16:46 even if it’s few points — it may indicate a localized severe failure on that path.

One-line recap:
- 12 metric segments checked → 736 matching trace rows found → 52 trace rows exceeded their global P95; top priorities: ts-order-service (latency), ts-travel-service (CPU/durations), and the high-ratio ts-basic → ts-train trace.

The original code execution output of IPython Kernel is also provided below for reference:

(      segment_cmdb_id segment_kpi_name         segment_start_utc       trace_timestamp_utc          trace_cmdb_id                                 trace_name  trace_value  global_P95  trace_value_above_P95  trace_max_vs_threshold_ratio
0     ts-auth-service         workload  2024-01-23 16:32:00+0000  2024-01-23 16:33:00+0000        ts-auth-service                  trace.from_root.row_count    82.000000   79.000000                   True                      1.037975
1     ts-auth-service         workload  2024-01-23 16:32:00+0000  2024-01-23 16:32:00+0000        ts-auth-service                  trace.from_root.row_count    79.000000   79.000000                  False                      1.000000
2     ts-auth-service         workload  2024-01-23 16:32:00+0000  2024-01-23 16:32:00+0000        ts-auth-service               trace.from_root.duration_p95     1.159416    1.204133                  False                      0.962863
3     ts-auth-service         workload  2024-01-23 16:32:00+0000  2024-01-23 16:33:00+0000        ts-auth-service         trace.to_ts-auth-service.row_count   328.000000  342.200000                  False                      0.958504
4     ts-auth-service         workload  2024-01-23 16:32:00+0000  2024-01-23 16:33:00+0000        ts-auth-service       trace.from_ts-auth-service.row_count   328.000000  344.000000                  False                      0.953488
..                ...              ...                       ...                       ...                    ...                                        ...          ...         ...                    ...                           ...
45  ts-travel-service              cpu  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000                   root    trace.to_ts-travel-service.duration_p95     0.644348    0.915168                  False                      0.704077
46  ts-travel-service              cpu  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000      ts-travel-service   trace.from_ts-food-service.duration_mean     0.009382    0.013575                  False                      0.691122
47  ts-travel-service              cpu  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000      ts-travel-service  trace.from_ts-travel-service.duration_p95     0.045335    0.066676                  False                      0.679933
48  ts-travel-service              cpu  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000  ts-ticketinfo-service  trace.from_ts-travel-service.duration_p95     0.044337    0.066676                  False                      0.664958
49  ts-travel-service              cpu  2024-01-23 16:38:00+0000  2024-01-23 16:38:00+0000      ts-travel-service      trace.to_ts-seat-service.duration_p95     0.064736    0.099228                  False                      0.652394

[50 rows x 10 columns], 'Total segments processed: 12. Total matching trace rows found: 736. Trace rows with value > global_P95: 52.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse existing log_df if present; otherwise load and parse
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# 1) Parse timestamp as UTC and ensure numeric value
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 2) Compute global P95 for each distinct log_name using the full series (not window-restricted)
log_global_p95 = (
    log_df
    .groupby('log_name', as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_P95'})
)

# 3) Filter log rows to the incident window and cmdb_id == 'ts-order-service'
incident_start = pd.to_datetime("2024-01-23 16:29:06", utc=True)
incident_end   = pd.to_datetime("2024-01-23 16:59:06", utc=True)

ts_order_logs_window = log_df[
    (log_df['cmdb_id'] == 'ts-order-service') &
    (log_df['timestamp'] >= incident_start) &
    (log_df['timestamp'] <= incident_end)
].copy()

# 4) For each log_name present for ts-order-service, produce summary row
if ts_order_logs_window.empty:
    summary_table = pd.DataFrame(columns=[
        'log_name','global_P95','points_in_window','points_above_P95',
        'earliest_timestamp_above_P95','max_value_in_window','error_timestamps_up_to_5'
    ])
else:
    # Aggregate per log_name within window
    agg = (
        ts_order_logs_window
        .groupby('log_name', as_index=False)
        .agg(
            points_in_window = ('value','count'),
            max_value_in_window = ('value','max')
        )
    )

    # Attach global P95 (calculated globally across log.csv)
    summary = agg.merge(log_global_p95, on='log_name', how='left')

    # Compute points_above_P95 and earliest timestamp above P95 per log_name
    def compute_above_info(g):
        name = g.name
        gp95 = summary.loc[summary['log_name'] == name, 'global_P95'].values[0]
        if pd.isna(gp95):
            pts_above = 0
            earliest = pd.NaT
        else:
            mask = g['value'] > gp95
            pts_above = int(mask.sum())
            earliest = g.loc[mask, 'timestamp'].min() if pts_above > 0 else pd.NaT
        return pd.Series({
            'points_above_P95': pts_above,
            'earliest_timestamp_above_P95': earliest
        })

    above_info = ts_order_logs_window.groupby('log_name').apply(compute_above_info).reset_index()

    summary = summary.merge(above_info, on='log_name', how='left')

    # Format earliest timestamps and prepare error timestamps list (up to 5) for log.error_count
    summary['earliest_timestamp_above_P95'] = pd.to_datetime(summary['earliest_timestamp_above_P95'])
    summary['earliest_timestamp_above_P95'] = summary['earliest_timestamp_above_P95'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    summary['earliest_timestamp_above_P95'] = summary['earliest_timestamp_above_P95'].where(summary['earliest_timestamp_above_P95'].notna(), pd.NA)

    # Prepare error timestamps: find up to 5 timestamps where log_name == 'log.error_count' and value > 0
    error_ts_list = []
    error_rows = ts_order_logs_window[ts_order_logs_window['log_name'] == 'log.error_count']
    error_positive = error_rows[error_rows['value'] > 0].sort_values('timestamp')
    # format up to 5 timestamps as comma-separated strings
    if not error_positive.empty:
        formatted = error_positive['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z').tolist()[:5]
        error_ts_str = ", ".join(formatted)
    else:
        error_ts_str = pd.NA

    # Add column error_timestamps_up_to_5 where only log.error_count row gets the list
    def error_ts_for_row(log_name):
        return error_ts_str if log_name == 'log.error_count' else pd.NA

    summary['error_timestamps_up_to_5'] = summary['log_name'].apply(error_ts_for_row)

    # Select and order columns
    summary_table = summary[[
        'log_name',
        'global_P95',
        'points_in_window',
        'points_above_P95',
        'earliest_timestamp_above_P95',
        'max_value_in_window',
        'error_timestamps_up_to_5'
    ]].sort_values(by='points_in_window', ascending=False).reset_index(drop=True)

# 5) One-line summary: total log rows for ts-order-service in window and how many had value > global_P95
total_rows_ts_order = int(ts_order_logs_window.shape[0])
# To count how many rows had value > global_P95, merge per-row global_P95 by log_name and count
if ts_order_logs_window.empty:
    rows_above_count = 0
else:
    per_row = ts_order_logs_window.merge(log_global_p95, on='log_name', how='left')
    per_row['above_global_P95'] = per_row['value'] > per_row['global_P95']
    rows_above_count = int(per_row['above_global_P95'].sum())

one_line_summary = f"Total ts-order-service log rows in window: {total_rows_ts_order}. Rows with value > global_P95: {rows_above_count}."

# Store variables in kernel for later inspection:
# - ts_order_logs_window
# - log_global_p95
# They are already assigned above.

# Final compact outputs: summary_table and one_line_summary
summary_table, one_line_summary

```
Out[7]:
```
Summary (plain English):

- What I checked: parsed logs and computed global P95 per log_name using the entire log.csv, then inspected ts-order-service logs in the incident window 2024-01-23 16:29:06–16:59:06 UTC.

- Key results:
  - Total ts-order-service log rows in the window: 50 (two log_names, 25 points each).
  - log.error_count: global P95 = 0.0, points_in_window = 25, max value = 0 → no errors observed and no timestamps with error_count > 0.
  - log.row_count: global P95 = 1881.2, points_in_window = 25, max value in window = 527 → below the global P95, so no row_count anomaly by the global-P95 rule.
  - Overall: 0 log rows for ts-order-service exceeded their global P95 in the incident window.

- Implication:
  - ts-order-service logs do not show error spikes or log-based anomalies during the incident window. The metric/trace anomalies already identified for ts-order-service (latency and increased incoming trace counts) are not mirrored by elevated error_count or abnormal log.row_count in these logs.
  - Next steps: focus on trace/metric correlation (already highlighted) and upstream dependencies or infra (DB, network) since logs on the service itself show no error evidence. If needed, enable more detailed application logging for the timestamps of the metric/trace spikes.

The original code execution output of IPython Kernel is also provided below for reference:

(          log_name  global_P95  points_in_window  points_above_P95 earliest_timestamp_above_P95  max_value_in_window error_timestamps_up_to_5
0  log.error_count         0.0                25                 0                         <NA>                    0                     <NA>
1    log.row_count      1881.2                25                 0                         <NA>                  527                     <NA>, 'Total ts-order-service log rows in window: 50. Rows with value > global_P95: 0.')```
```